<a href="https://colab.research.google.com/github/nam179/Lab01_2351260677_TranVanNam/blob/main/Untitled0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install numpy scipy soundfile librosa

In [6]:
import os
import numpy as np
import scipy.signal as signal
import soundfile as sf
import librosa
import urllib.request

# Thông tin sinh viên
STUDENT_NAME = "Trần Văn Nam"
STUDENT_ID = "2351260677"
GITHUB_USER = "namphwong"  # Tên Username GitHub của bạn

os.makedirs("audio", exist_ok=True)
os.makedirs("figures", exist_ok=True)

print(f"=== BÀI THỰC HÀNH LAB 1 - CSE457 ===")
print(f"Sinh viên: {STUDENT_NAME} | MSV: {STUDENT_ID}\n")

# Tự động tải file ghi âm từ GitHub về Colab
url = f"https://raw.githubusercontent.com/{GITHUB_USER}/Lab01_{STUDENT_ID}_TranVanNam/main/speech_input.m4a"
input_path = "speech_input.m4a"

try:
    urllib.request.urlretrieve(url, input_path)
    print("-> Đã tải thành công file ghi âm từ GitHub về Colab!")
except Exception as e:
    print("Lỗi tải file:", e)

# 1. Khối A: Đọc và Chuẩn hóa
x_raw, Fs = librosa.load(input_path, sr=None, mono=False)
channels = x_raw.shape[0] if x_raw.ndim > 1 else 1
x_mono = np.mean(x_raw, axis=0) if channels > 1 else x_raw
duration = len(x_mono) / Fs
x = x_mono / np.max(np.abs(x_mono))

print("\n--- KHỐI A: METADATA ---")
print(f"Fs: {Fs} Hz | Channels: {channels} | Duration: {duration:.3f} s")
sf.write("audio/speech_mono_normalized.wav", x, Fs)

# 2. Khối B: Miền thời gian
peak_val = np.max(np.abs(x))
rms_val = np.sqrt(np.mean(x**2))
energy_val = np.sum(x**2)
print(f"\n--- KHỐI B ---")
print(f"Peak: {peak_val:.4f} | RMS: {rms_val:.4f} | Energy: {energy_val:.2f}")

# 3. Khối C: FFT
start_idx, end_idx = int(0.5 * Fs), int(1.0 * Fs)
seg = x[start_idx:end_idx]
w_ham = np.hamming(len(seg))
NFFT = 65536
X = np.fft.rfft(seg * w_ham, n=NFFT)
freqs = np.fft.rfftfreq(NFFT, 1/Fs)
mag_db = 20 * np.log10(np.maximum(np.abs(X), 1e-12))
df = Fs / NFFT
print(f"\n--- KHỐI C: FFT bin df = {df:.3f} Hz ---")

# 4. Khối F: Lọc FIR Low-pass
numtaps = 201
cutoff_hz = 2000.0
b_fir = signal.firwin(numtaps, cutoff=cutoff_hz, fs=Fs, window='hamming')
y_filtered = signal.lfilter(b_fir, [1.0], x)
sf.write("audio/filtered_speech_lpf2k.wav", y_filtered, Fs)
print(f"\n--- KHỐI F: Đã lọc LPF 2kHz ---")

# 5. Khối G: Lượng tử hóa & Resampling
def quantize(x_signal, B):
    qmax = 2**(B - 1) - 1
    return np.round(np.clip(x_signal, -1, 1) * qmax) / qmax

def snr_db(x_orig, x_quant):
    e = x_quant - x_orig
    return 10 * np.log10(np.sum(x_orig**2) / np.sum(e**2))

print("\n--- KHỐI G: SNR LƯỢNG TỬ HÓA ---")
for B in [4, 6, 8, 12, 16]:
    x_q = quantize(x, B)
    print(f"Bit B = {B:2d} | SNR = {snr_db(x, x_q):6.2f} dB")
    sf.write(f"audio/quantized_speech_{B}bit.wav", x_q, Fs)

x_16k = librosa.resample(x, orig_sr=Fs, target_sr=16000)
sf.write("audio/resampled_speech_16k.wav", x_16k, 16000)
print("\n-> HOÀN THÀNH TOÀN BỘ BÀI LAB 1!")

=== BÀI THỰC HÀNH LAB 1 - CSE457 ===
Sinh viên: Trần Văn Nam | MSV: 2351260677

Lỗi tải file: HTTP Error 404: Not Found


/tmp/ipykernel_705/1383929728.py:30: UserWarning: PySoundFile failed. Trying audioread instead.
  x_raw, Fs = librosa.load(input_path, sr=None, mono=False)



--- KHỐI A: METADATA ---
Fs: 48000 Hz | Channels: 2 | Duration: 2.944 s

--- KHỐI B ---
Peak: 1.0000 | RMS: 0.1748 | Energy: 4317.02

--- KHỐI C: FFT bin df = 0.732 Hz ---

--- KHỐI F: Đã lọc LPF 2kHz ---

--- KHỐI G: SNR LƯỢNG TỬ HÓA ---
Bit B =  4 | SNR =  12.75 dB
Bit B =  6 | SNR =  25.60 dB
Bit B =  8 | SNR =  37.80 dB
Bit B = 12 | SNR =  61.94 dB
Bit B = 16 | SNR =  86.01 dB

-> HOÀN THÀNH TOÀN BỘ BÀI LAB 1!
